# Chapter 4

In [1]:
import torch
from torch import tensor
import numpy as np
import matplotlib.pyplot as plt
from fastai.vision.all import *
from fastbook import *

## Loss functions: MSE

- `mse` = *mean squared error*: one single number that says how far off my predictions are on average.
- Three steps, inside out: `preds - targets` (per-item error) → `**2` (make positive, punish big errors harder) → `.mean()` (average everything into one scalar).
- `.mean()` exists on **tensors/arrays**, not on plain Python ints — that's why `mse(5, 3)` failed with `AttributeError`.
- `tensor` comes from `from fastai.vision.all import *` (or `from torch import tensor`) — a `NameError` means the import cell wasn't run in this kernel.
- Use floats (`5.` not `5`): the mean of integer tensors raises an error.
- Training needs *one scalar* loss so the optimizer knows which direction makes the model better.

In [2]:
def mse(preds, targets): return ((preds - targets)**2).mean()

preds   = tensor([4.0, 4.0, 6.0, 2.0])
targets = tensor([3.0, 4.0, 2.0, 1.0])

print('errors:        ', preds - targets)
print('squared errors:', (preds - targets)**2)
print('mse:           ', mse(preds, targets))
print('rmse:          ', mse(preds, targets).sqrt())  # back in original units

errors:         tensor([1., 0., 4., 1.])
squared errors: tensor([ 1.,  0., 16.,  1.])
mse:            tensor(4.5000)
rmse:           tensor(2.1213)


In [3]:
# Experiment: what happens if one prediction is WAY off?
# Big errors dominate MSE because of the squaring.
mse(tensor([2.0, 4.0, 20.0, 1.0]), targets)

tensor(81.2500)

## Pixel similarity

In [4]:

img = Image.open('my_images/3.png')
img

In [5]:
array(img)

array([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   2,  13,  42, 133,  93, 163,  93, 244, 186,  71,  13,  68,   0,   0,  51,  25,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,  37, 253, 253, 253, 253, 253, 253, 253, 178, 191, 238, 201,   7

In [6]:
tensor(img)

tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   2,  13,  42, 133,  93, 163,  93, 244, 186,  71,  13,  68,   0,   0,  51,  25,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,  37, 253, 253, 253, 253, 253, 253, 253, 178, 191, 238, 2

In [7]:
a = tensor(1,2,3)
b = tensor(5,6,7)
t = [a,b]
tensors = torch.stack(t)

In [8]:
tensors.shape # dimension of the new matrix

torch.Size([2, 3])

In [9]:
tensors

tensor([[1, 2, 3],
        [5, 6, 7]])

In [10]:
# length of tensor's shape --> tensor rank
len(tensors.shape), tensors.ndim # both the same

(2, 2)